<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session06_regularization_and_normalization_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۶: منظم‌سازی و نرمال‌سازی: Dropout، Weight Decay، BatchNorm و Data Augmentation

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session06_regularization_and_normalization.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

MEAN, STD = 0.2860, 0.3530
base_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])
aug_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,)),
])
full_train = datasets.FashionMNIST('data', train=True, download=True, transform=base_tf)
aug_full = datasets.FashionMNIST('data', train=True, download=True, transform=aug_tf)

perm = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(42))
small_idx, val_idx = perm[:5_000], perm[50_000:]          # same split as the session notebook
small_loader = DataLoader(Subset(full_train, small_idx), batch_size=128, shuffle=True, num_workers=2)
aug_loader = DataLoader(Subset(aug_full, small_idx), batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(full_train, val_idx), batch_size=1000, num_workers=2)

def train_one_epoch(model, loader, criterion, optimizer, device, scheduler=None):
    """One pass over the training data. `scheduler` is stepped after every batch if given."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, optimizer, epochs, device, scheduler=None, verbose=True):
    """Train for `epochs` epochs and return the history of losses and accuracies."""
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
        va = evaluate(model, val_loader, criterion, device)
        for k, v in zip(history, (*tr, *va)):
            history[k].append(v)
        if verbose:
            print(f'epoch {epoch:2d} | train loss {tr[0]:.4f} acc {tr[1]:.4f} | val loss {va[0]:.4f} acc {va[1]:.4f}')
    return history


def make_mlp(hidden=1024, depth=3, dropout=0.0, batchnorm=False, layernorm=False):
    layers, n_in = [nn.Flatten()], 784
    for _ in range(depth):
        layers.append(nn.Linear(n_in, hidden))
        if batchnorm:
            layers.append(nn.BatchNorm1d(hidden))
        if layernorm:
            layers.append(nn.LayerNorm(hidden))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        n_in = hidden
    layers.append(nn.Linear(n_in, 10))
    return nn.Sequential(*layers)


EPOCHS = 40

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

آزمایش Dropout را با $p \in \{0.1, 0.3, 0.5, 0.7\}$ تکرار کنید و بهترین دقت اعتبارسنجی را بر حسب $p$ رسم کنید.

</div>

In [ ]:
best_acc = {}
for p in (0.1, 0.3, 0.5, 0.7):
    torch.manual_seed(0)
    model = make_mlp(dropout=p).to(device)
    h = fit(model, small_loader, val_loader, torch.optim.Adam(model.parameters(), lr=1e-3), EPOCHS, device, verbose=False)
    best_acc[p] = max(h['val_acc'])
    print(f'p = {p}: best val acc {best_acc[p]:.4f} | final train acc {h["train_acc"][-1]:.4f}')

plt.plot(list(best_acc), list(best_acc.values()), 'o-')
plt.xlabel('dropout probability p'); plt.ylabel('best validation accuracy'); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

منحنی معمولاً شکل یک «تپه» دارد: با $p$ کم منظم‌سازی ناکافی است و مدل باز هم بیش‌برازش می‌کند؛ با $p$ خیلی زیاد (۰٫۷) در هر گام بخش بزرگی از شبکه خاموش است، یادگیری کند می‌شود و دقت آموزش هم پایین می‌ماند. مقدار بهینه برای این شبکهٔ بزرگ و دادهٔ کوچک معمولاً در محدودهٔ ۰٫۳ تا ۰٫۵ است. برای شبکه‌ای که داده‌اش زیاد است، $p$ بهینه کوچک‌تر می‌شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

Weight Decay را با $\lambda \in \{0.001, 0.01, 0.1, 1\}$ امتحان کنید. با $\lambda$ خیلی بزرگ چه اتفاقی می‌افتد (Underfitting)؟

</div>

In [ ]:
for lam in (0.001, 0.01, 0.1, 1.0):
    torch.manual_seed(0)
    model = make_mlp().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=lam)
    h = fit(model, small_loader, val_loader, opt, EPOCHS, device, verbose=False)
    w_norm = sum(p.pow(2).sum() for n, p in model.named_parameters() if 'weight' in n).sqrt().item()
    print(f'lambda = {lam:<5}: final train acc {h["train_acc"][-1]:.4f} | best val acc {max(h["val_acc"]):.4f} '
          f'| best val loss {min(h["val_loss"]):.4f} | weight norm {w_norm:7.2f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با بزرگ‌تر شدن $\lambda$، نُرم وزن‌ها کوچک‌تر می‌شود. در AdamW هر گام وزن‌ها را در $(1 - \eta\lambda)$ ضرب می‌کند؛ با $\lambda = 1$ و $\eta = 10^{-3}$ این ضریب $0.999$ است. پس در ۴۰ Epoch (حدود ۱۶۰۰ گام) وزن‌ها چند برابر کوچک می‌شوند. برای Weight Decay بسیار بزرگ، جریمه بر دادهٔ آموزش غالب می‌شود: مدل بیش از حد ساده می‌ماند و **دقت آموزش هم** پایین می‌آید؛ نشانهٔ کلاسیک **Underfitting**. نکته: در AdamW اثر $\lambda$ در ضرب با نرخ یادگیری معنا دارد، پس مقادیر مناسبش از SGD متفاوت است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

در مدل `batchnorm`، بعد از آموزش یک بار ارزیابی را **بدون** `model.eval()` انجام دهید (مدل در حالت train بماند) و با Batchهای اندازه ۲ پیش‌بینی کنید. چرا دقت افت می‌کند؟

</div>

In [ ]:
torch.manual_seed(0)
model = make_mlp(batchnorm=True).to(device)
fit(model, small_loader, val_loader, torch.optim.Adam(model.parameters(), lr=1e-3), EPOCHS, device, verbose=False)
tiny_loader = DataLoader(Subset(full_train, val_idx), batch_size=2)


@torch.no_grad()
def accuracy(model, loader):
    correct = n = 0
    for X, y in loader:
        correct += (model(X.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
    return correct / n


model.eval()
print(f'eval mode,  batch size 2   : {accuracy(model, tiny_loader):.4f}')
model.train()                                     # BatchNorm now uses the statistics of each batch of 2
print(f'train mode, batch size 2   : {accuracy(model, tiny_loader):.4f}')
print(f'train mode, batch size 1000: {accuracy(model, val_loader):.4f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در حالت `train`، BatchNorm هر ویژگی را با میانگین و واریانس **همان Batch** نرمال می‌کند. با Batch دوتایی این آمار بسیار نویزی است. خروجی هر نمونه هم به نمونهٔ دیگری بستگی پیدا می‌کند که اتفاقاً با آن در یک Batch افتاده است: برای هر ویژگی، دو مقدار نرمال‌شده همیشه $\pm 1$ می‌شوند و اطلاعات مقدار واقعی از بین می‌رود. پس دقت شدیداً افت می‌کند. با Batch بزرگ آمار به آمار کل داده نزدیک است و افت کمتر است، ولی پیش‌بینی هنوز به ترکیب Batch وابسته است. در حالت `eval`، BatchNorm از **Running Statistics** ذخیره‌شده در آموزش استفاده می‌کند و خروجی هر نمونه مستقل و قطعی است. (این آزمایش آمار ذخیره‌شده را هم تغییر می‌دهد، چون در حالت train به‌روز می‌شوند.)

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

`nn.LayerNorm` را جایگزین `nn.BatchNorm1d` در `make_mlp` کنید و نتایج دو آزمایش BatchNorm را تکرار کنید.

</div>

In [ ]:
perm_full = perm[:50_000]
train_loader = DataLoader(Subset(full_train, perm_full), batch_size=128, shuffle=True, num_workers=2)
results = {}
for name, kw in {'BatchNorm': dict(batchnorm=True), 'LayerNorm': dict(layernorm=True)}.items():
    # experiment 1: regularisation on 5,000 images (big MLP, Adam)
    torch.manual_seed(0)
    m = make_mlp(**kw).to(device)
    h1 = fit(m, small_loader, val_loader, torch.optim.Adam(m.parameters(), lr=1e-3), EPOCHS, device, verbose=False)
    # experiment 2: optimisation of a deep 8-layer MLP with SGD for 3 epochs on 50,000 images
    torch.manual_seed(0)
    m = make_mlp(hidden=256, depth=8, **kw).to(device)
    h2 = fit(m, train_loader, val_loader, torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9), 3, device, verbose=False)
    print(f'{name}: (1) best val acc {max(h1["val_acc"]):.4f}, best val loss {min(h1["val_loss"]):.4f} | '
          f'(2) deep MLP val acc per epoch {np.round(h2["val_acc"], 4)}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

LayerNorm در هر دو آزمایش نتیجه‌ای نزدیک به BatchNorm می‌دهد: آموزش شبکهٔ عمیق را مثل BatchNorm پایدار می‌کند. چون آمارش را از ویژگی‌های **هر نمونه** می‌گیرد، به اندازهٔ Batch وابسته نیست و در آموزش و ارزیابی دقیقاً یک رفتار دارد (مشکل تمرین ۳ را ندارد). از طرف دیگر، نویز آماری Batch در BatchNorm کمی اثر منظم‌سازی دارد که LayerNorm ندارد. به همین دلیل در CNNها هنوز BatchNorm رایج است و در Transformerها LayerNorm.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

مدل‌های `augmentation` و `all combined` را ۱۵۰ Epoch آموزش دهید. آیا در نهایت از Dropout بهتر می‌شوند؟

</div>

In [ ]:
long = {}
for name, kw, opt_fn in [
    ('dropout 0.5', dict(dropout=0.5), lambda p: torch.optim.Adam(p, lr=1e-3)),
    ('augmentation', dict(), lambda p: torch.optim.Adam(p, lr=1e-3)),
    ('all combined', dict(dropout=0.3, batchnorm=True), lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=0.05)),
]:
    torch.manual_seed(0)
    m = make_mlp(**kw).to(device)
    loader = small_loader if name == 'dropout 0.5' else aug_loader
    long[name] = fit(m, loader, val_loader, opt_fn(m.parameters()), 150, device, verbose=False)
    print(f'{name:13}: val acc at epoch 40 {long[name]["val_acc"][39]:.4f} | at epoch 150 {long[name]["val_acc"][-1]:.4f} '
          f'| best {max(long[name]["val_acc"]):.4f}')

for name, h in long.items():
    plt.plot(h['val_acc'], label=name)
plt.axvline(40, color='gray', ls=':'); plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.legend(); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

Augmentation مسئلهٔ آموزش را **سخت‌تر** می‌کند: مدل در هر Epoch نسخهٔ متفاوتی از هر تصویر می‌بیند، پس دیرتر به سقف می‌رسد. در ۴۰ Epoch هنوز در حال یادگیری بود (Underfit)؛ با آموزش طولانی‌تر، فاصلهٔ آن با Dropout کم یا برعکس می‌شود. در مقابل، مدل Dropout روی ۵۰۰۰ تصویر ثابت، با Epochهای بیشتر چیز جدیدی نمی‌بیند. **درس اصلی:** روش‌های منظم‌سازی قوی‌تر، بودجهٔ آموزش بیشتری می‌خواهند و مقایسهٔ منصفانه باید با Epoch کافی برای هر روش انجام شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۶

آزمایش مقایسه نهایی را با ۵۰٬۰۰۰ تصویر آموزشی تکرار کنید. آیا تفاوت روش‌ها کمتر می‌شود؟ چرا؟

</div>

In [ ]:
big_loader = DataLoader(Subset(full_train, perm[:50_000]), batch_size=128, shuffle=True, num_workers=2)
big_aug_loader = DataLoader(Subset(aug_full, perm[:50_000]), batch_size=128, shuffle=True, num_workers=2)
configs = {
    'baseline': (dict(), big_loader, lambda p: torch.optim.Adam(p, lr=1e-3)),
    'weight decay': (dict(), big_loader, lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=0.1)),
    'dropout 0.5': (dict(dropout=0.5), big_loader, lambda p: torch.optim.Adam(p, lr=1e-3)),
    'batchnorm': (dict(batchnorm=True), big_loader, lambda p: torch.optim.Adam(p, lr=1e-3)),
    'augmentation': (dict(), big_aug_loader, lambda p: torch.optim.Adam(p, lr=1e-3)),
}
for name, (kw, loader, opt_fn) in configs.items():
    torch.manual_seed(0)
    m = make_mlp(**kw).to(device)
    h = fit(m, loader, val_loader, opt_fn(m.parameters()), 15, device, verbose=False)
    print(f'{name:13}: best val acc {max(h["val_acc"]):.4f} | final train-val gap {h["train_acc"][-1] - h["val_acc"][-1]:+.4f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با ده برابر داده، دقت همهٔ روش‌ها بالاتر می‌رود و **فاصلهٔ آن‌ها کمتر می‌شود**. خود داده بهترین منظم‌ساز است: با نمونه‌های بیشتر، حفظ کردن نویز تک‌تک نمونه‌ها سخت‌تر و یادگیری الگوی عمومی سودمندتر می‌شود. منظم‌سازی وقتی بیشترین اثر را دارد که نسبت پارامتر به داده بالا باشد (در جلسه حدود ۲ میلیون پارامتر برای ۵۰۰۰ تصویر). برای تعداد Epoch کمتر (۱۵) هم این آزمایش روی GPU چند دقیقه طول می‌کشد.

</div>